# Motorsport: Lap-by-Lap Telemetry Analysis

Analyse telemetry data across multiple laps to identify performance trends, compare lap times, and find the fastest segments.

**Context:** In motorsport, sessions contain laps defined by trigger sources (e.g. timing beacons). Each lap has a start time and number, allowing per-lap data extraction and comparison.

In [ ]:
import sys
sys.path.insert(0, '..')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameter, get_laps
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")

SESSION_GUID = "a8b2b3b4-5530-4825-a5d1-bda63732361e"
SPEED_PARAM = "Speed:Shaft"          # Vehicle speed parameter
TEMP_PARAM = "Temperature:Sensors"   # Brake disc temperature

cs = f"DbEngine=SQLite;Data Source={os.path.join(DATA_DIR, 'motorsport-lap-analysis.ssn2')};"
sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID, connection_string=cs)
session_summary(session)

## Lap boundaries

In [ ]:
df_laps = get_laps(session)
print(f"{len(df_laps)} lap(s) found")
display(df_laps)

## Extract per-lap data

Read the speed parameter for each lap window.

In [ ]:
lap_data = {}
lap_times = df_laps["start_time_ns"].tolist()

for i in range(len(lap_times)):
    t_start = lap_times[i]
    t_end = lap_times[i + 1] if i + 1 < len(lap_times) else session.EndTime

    data = extract_parameter(session, SPEED_PARAM, t_start, t_end)
    if len(data) > 0:
        # Convert to relative time within lap
        rel_time = (data.index - data.index[0]) / 1e9
        lap_data[i] = pd.Series(data.values, index=rel_time)
        duration = rel_time[-1]
        print(f"  Lap {i}: {len(data)} samples, {duration:.2f} s")

print(f"\nExtracted {len(lap_data)} laps.")

## Overlay lap traces

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for lap_num, data in lap_data.items():
    ax.plot(data.index, data.values, label=f"Lap {lap_num}",
            linewidth=0.8, alpha=0.7)

ax.set_xlabel("Lap distance / time (s)")
ax.set_ylabel(SPEED_PARAM)
ax.set_title("Lap Overlay")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Per-lap statistics

In [ ]:
stats = []
for lap_num, data in lap_data.items():
    stats.append({
        "Lap": lap_num,
        "Duration (s)": data.index[-1],
        "Mean Speed": data.mean(),
        "Max Speed": data.max(),
        "Min Speed": data.min(),
        "Std Speed": data.std(),
    })

df_stats = pd.DataFrame(stats).set_index("Lap")
display(df_stats)

fastest = df_stats["Duration (s)"].idxmin()
print(f"\nFastest lap: {fastest} ({df_stats.loc[fastest, 'Duration (s)']:.2f} s)")

In [ ]:
client_session.Dispose()
print("Session closed.")